# GroupDNA - WhatsApp Group Chat Analyzer

    Built by Prithika | The Unlox Academy

---

## Introduction

GroupDNA is a WhatsApp group analytics project built using only **Python fundamentals, NumPy, and datetime**.  
The goal is to parse a raw exported WhatsApp chat file and generate a polished personality and activity report for the group.

---

## Objective

This notebook analyzes the chat dataset `hostel_bois.txt` and produces:

- group overview statistics
- activity trends
- busiest day and hour
- a NumPy-based activity heatmap
- top word frequency analysis
- response speed analysis
- silent streak detection
- message style metrics
- personality archetype assignment
- a final formatted **GroupDNA Report**

---

## Constraints Followed

- No pandas
- No matplotlib
- No seaborn
- No regex
- No `collections.Counter`
- Only Python fundamentals + NumPy + datetime


In [150]:
# Feature 0: Imports and configuration

import numpy as np
from datetime import datetime, timedelta
import string


In [151]:
# Feature 0.1: Helper functions

def line_starts_with_date(line):
    """
    Detects whether a line starts with the pattern DD/MM/YY
    using only string checks, no regex.
    """
    if len(line) < 8:
        return False

    pattern = line[:8]
    return (
        pattern[0:2].isdigit() and
        pattern[2] == '/' and
        pattern[3:5].isdigit() and
        pattern[5] == '/' and
        pattern[6:8].isdigit()
    )


def safe_split_once(text, separator):
    """
    Splits only once. Returns None if separator not found.
    """
    if separator not in text:
        return None
    return text.split(separator, 1)


def format_date(dt_obj):
    return dt_obj.strftime('%d %B %Y')


def format_short_date(dt_obj):
    return dt_obj.strftime('%d %b')


def minutes_to_readable(minutes):
    if minutes is None:
        return "N/A"

    if minutes < 60:
        return f"{minutes:.1f} minutes"

    hours = minutes / 60
    if hours < 24:
        return f"{hours:.1f} hours"

    days = hours / 24
    return f"{days:.1f} days"


def print_divider(width=60, char='='):
    print(char * width)


def make_bar(value, max_value, width=20, char='█'):
    if max_value == 0:
        return '.'
    filled = int(round((value / max_value) * width))
    if filled == 0 and value > 0:
        filled = 1
    return char * filled if filled > 0 else '.'


def clean_word(word):
    """
    Lowercase and strip punctuation from both ends.
    """
    return word.lower().strip(string.punctuation + '“”‘’"\'')


In [152]:
# Feature 1: File reading + parser

def read_chat_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.readlines()


def merge_multiline_messages(raw_lines):
    """
    Merges continuation lines into previous WhatsApp line.
    """
    merged = []

    for raw_line in raw_lines:
        line = raw_line.rstrip('\n')

        if not line.strip():
            continue

        if line_starts_with_date(line):
            merged.append(line)
        else:
            if merged:
                merged[-1] += ' ' + line.strip()
            else:
                # Rare edge case: continuation before first valid line
                continue

    return merged


def parse_chat(file_path):
    raw_lines = read_chat_file(file_path)
    merged_lines = merge_multiline_messages(raw_lines)

    messages = []
    system_count = 0
    media_count = 0
    deleted_count = 0
    group_name = "Unknown Group"

    for line in merged_lines:
        first_split = safe_split_once(line, ' - ')
        if first_split is None:
            system_count += 1
            continue

        timestamp_str, rest = first_split

        second_split = safe_split_once(rest, ': ')

        # System message: no sender/message split
        if second_split is None:
            system_count += 1

            # Try extracting group name from group creation line
            if 'created group "' in rest and '"' in rest:
                start = rest.find('created group "') + len('created group "')
                end = rest.rfind('"')
                if start < end:
                    group_name = rest[start:end]
            continue

        sender, text = second_split

        try:
            dt = datetime.strptime(timestamp_str, '%d/%m/%y, %H:%M')
        except ValueError:
            system_count += 1
            continue

        is_media = text.strip() == '<Media omitted>'
        is_deleted = text.strip() == 'This message was deleted'

        if is_media:
            media_count += 1
        if is_deleted:
            deleted_count += 1

        msg = {
            'timestamp_str': timestamp_str,
            'datetime': dt,
            'date': dt.date(),
            'time': dt.time(),
            'hour': dt.hour,
            'sender': sender.strip(),
            'text': text.strip(),
            'is_media': is_media,
            'is_deleted': is_deleted
        }
        messages.append(msg)

    return {
        'group_name': group_name,
        'messages': messages,
        'system_count': system_count,
        'media_count': media_count,
        'deleted_count': deleted_count,
        'merged_line_count': len(merged_lines),
        'raw_line_count': len(raw_lines)
    }


In [153]:
# Feature 1.1: Run parser
file_path = "/content/hostel_bois.txt"
chat_data = parse_chat(file_path)

group_name = chat_data['group_name']
messages = chat_data['messages']
system_count = chat_data['system_count']
media_count = chat_data['media_count']
deleted_count = chat_data['deleted_count']

print("Parsing complete.")
print(f"Group name         : {group_name}")
print(f"Raw lines          : {chat_data['raw_line_count']}")
print(f"Merged lines       : {chat_data['merged_line_count']}")
print(f"Parsed messages    : {len(messages)}")
print(f"System messages    : {system_count}")
print(f"Media messages     : {media_count}")
print(f"Deleted messages   : {deleted_count}")

print("\nFirst 5 parsed messages:")
for msg in messages[:5]:
    print(msg)

print("\nLast 5 parsed messages:")
for msg in messages[-5:]:
    print(msg)


Parsing complete.
Group name         : Hostel Bois 4ever
Raw lines          : 3178
Merged lines       : 3178
Parsed messages    : 3174
System messages    : 4
Media messages     : 32
Deleted messages   : 15

First 5 parsed messages:
{'timestamp_str': '01/04/24, 01:17', 'datetime': datetime.datetime(2024, 4, 1, 1, 17), 'date': datetime.date(2024, 4, 1), 'time': datetime.time(1, 17), 'hour': 1, 'sender': 'Rahul', 'text': 'scene fix', 'is_media': False, 'is_deleted': False}
{'timestamp_str': '01/04/24, 01:17', 'datetime': datetime.datetime(2024, 4, 1, 1, 17), 'date': datetime.date(2024, 4, 1), 'time': datetime.time(1, 17), 'hour': 1, 'sender': 'Rahul', 'text': 'haan', 'is_media': False, 'is_deleted': False}
{'timestamp_str': '01/04/24, 01:18', 'datetime': datetime.datetime(2024, 4, 1, 1, 18), 'date': datetime.date(2024, 4, 1), 'time': datetime.time(1, 18), 'hour': 1, 'sender': 'Rahul', 'text': 'kya scene', 'is_media': False, 'is_deleted': False}
{'timestamp_str': '01/04/24, 02:13', 'dateti

In [154]:
# Feature 2: Group overview

def get_participants(messages):
    participants = set()
    for msg in messages:
        participants.add(msg['sender'])
    return sorted(list(participants))


def count_messages_per_person(messages):
    counts = {}
    for msg in messages:
        sender = msg['sender']
        counts[sender] = counts.get(sender, 0) + 1
    return counts


def get_date_range(messages):
    if not messages:
        return None, None, 0

    first_dt = messages[0]['datetime']
    last_dt = messages[-1]['datetime']

    total_days = (last_dt.date() - first_dt.date()).days + 1
    return first_dt, last_dt, total_days


def sort_counts_desc(counts_dict):
    return sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)


def print_group_overview(group_name, messages):
    participants = get_participants(messages)
    message_counts = count_messages_per_person(messages)
    sorted_counts = sort_counts_desc(message_counts)
    first_dt, last_dt, total_days = get_date_range(messages)
    total_messages = len(messages)

    print_divider()
    print("  GROUP OVERVIEW")
    print_divider()
    print(f"  Group           : {group_name}")
    print(f"  Period          : {format_date(first_dt)} to {format_date(last_dt)} ({total_days} days)")
    print(f"  Total messages  : {total_messages:,}")
    print(f"  Participants    : {len(participants)}")
    print()
    print("  MESSAGES PER PERSON")

    for name, count in sorted_counts:
        pct = (count / total_messages) * 100 if total_messages else 0
        print(f"    {name:<8}: {count:>4}  ({pct:>4.1f}%)")


In [155]:
# Feature 2.1: Display overview

print_group_overview(group_name, messages)


  GROUP OVERVIEW
  Group           : Hostel Bois 4ever
  Period          : 01 April 2024 to 30 May 2024 (60 days)
  Total messages  : 3,174
  Participants    : 6

  MESSAGES PER PERSON
    Rahul   :  953  (30.0%)
    Priya   :  718  (22.6%)
    Neha    :  635  (20.0%)
    Aman    :  490  (15.4%)
    Karan   :  354  (11.2%)
    Vikas   :   24  ( 0.8%)


In [156]:
# Feature 3: Most active day and hour

def get_busiest_day(messages):
    day_counts = {}
    for msg in messages:
        day = msg['date']
        day_counts[day] = day_counts.get(day, 0) + 1

    busiest_day = max(day_counts, key=day_counts.get)
    return busiest_day, day_counts[busiest_day], day_counts


def get_busiest_hour(messages):
    hour_counts = {}
    for hour in range(24):
        hour_counts[hour] = 0

    for msg in messages:
        hour_counts[msg['hour']] += 1

    busiest_hour = max(hour_counts, key=hour_counts.get)
    return busiest_hour, hour_counts[busiest_hour], hour_counts


def print_busiest_stats(messages):
    busiest_day, busiest_day_count, _ = get_busiest_day(messages)
    busiest_hour, busiest_hour_count, _ = get_busiest_hour(messages)

    print_divider()
    print("  ACTIVITY PEAKS")
    print_divider()
    print(f"  Busiest day     : {format_date(datetime.combine(busiest_day, datetime.min.time()))}  ({busiest_day_count} messages)")
    print(f"  Busiest hour    : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00  ({busiest_hour_count} messages)")


In [157]:
# Feature 3.1: Output

print_busiest_stats(messages)


  ACTIVITY PEAKS
  Busiest day     : 04 May 2024  (76 messages)
  Busiest hour    : 18:00 - 19:00  (248 messages)


In [158]:
# Feature 4: NumPy activity heatmap

def build_activity_heatmap(messages, participants):
    participant_index = {}
    for i, person in enumerate(participants):
        participant_index[person] = i

    heatmap = np.zeros((len(participants), 24), dtype=int)

    for msg in messages:
        row = participant_index[msg['sender']]
        col = msg['hour']
        heatmap[row][col] += 1

    return heatmap, participant_index


def cell_to_symbol(value, row_max):
    if value == 0:
        return '.'
    if row_max == 0:
        return '.'

    ratio = value / row_max

    if ratio <= 0.25:
        return '░'
    elif ratio <= 0.50:
        return '▒'
    else:
        return '█'


def print_activity_heatmap(heatmap, participants):
    print_divider()
    print("  ACTIVITY HEATMAP (messages by hour)")
    print_divider()

    header_hours = "      " + " ".join([f"{h:02d}" for h in range(24)])
    print(header_hours)

    for i, person in enumerate(participants):
        row = heatmap[i]
        row_max = int(np.max(row))
        symbols = [cell_to_symbol(v, row_max) for v in row]
        print(f"  {person:<8} " + " ".join(symbols))


In [159]:
# Feature 4.1: Run heatmap

participants = get_participants(messages)
heatmap, participant_index = build_activity_heatmap(messages, participants)

print("Heatmap shape:", heatmap.shape)
print_activity_heatmap(heatmap, participants)


Heatmap shape: (6, 24)
  ACTIVITY HEATMAP (messages by hour)
      00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
  Aman     █ █ █ █ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . █
  Karan    . . . . . . . ░ ▒ ▒ █ ▒ █ █ █ █ █ █ █ █ █ ▒ ░ ░
  Neha     . . . . . ▒ ░ ░ █ █ █ ▒ █ █ ▒ ░ █ █ █ █ █ ▒ ▒ ▒
  Priya    . . . . . . ░ ▒ █ █ █ █ █ █ █ ▒ ▒ █ █ █ █ ▒ ▒ ░
  Rahul    ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █
  Vikas    . . . . . . . ▒ █ ▒ ▒ . █ █ . ▒ ▒ █ █ █ ▒ ▒ ▒ █


In [160]:
# Feature 5: Top word analysis

STOP_WORDS = {
    'i', 'is', 'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
    'it', 'this', 'that', 'are', 'am', 'was', 'were', 'be', 'been', 'being',
    'you', 'your', 'we', 'our', 'they', 'their', 'them', 'he', 'she', 'his', 'her',
    'at', 'by', 'with', 'from', 'as', 'but', 'if', 'so', 'do', 'did', 'does',
    'have', 'has', 'had', 'will', 'would', 'should', 'can', 'could',
    'me', 'my', 'mine', 'us', 'him', 'hers', 'its',
    'na', 'hai', 'haan', 'abey', 'ok', 'okay'
}


def extract_words_from_text(text):
    words = []
    for token in text.split():
        word = clean_word(token)
        if not word:
            continue
        if word in STOP_WORDS:
            continue
        words.append(word)
    return words


def build_word_frequency(messages):
    word_counts = {}

    for msg in messages:
        if msg['is_media'] or msg['is_deleted']:
            continue

        words = extract_words_from_text(msg['text'])
        for word in words:
            word_counts[word] = word_counts.get(word, 0) + 1

    return word_counts


def get_top_n_items(counts_dict, n=10):
    items = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)
    return items[:n]


def print_top_words(word_counts, top_n=10):
    top_words = get_top_n_items(word_counts, top_n)
    max_count = top_words[0][1] if top_words else 0

    print_divider()
    print("  THIS GROUP'S FAVOURITE WORDS")
    print_divider()

    for word, count in top_words:
        bar = make_bar(count, max_count, width=20, char='█')
        print(f"  {word:<10} {bar:<20} {count}")


In [161]:
# Feature 5.1: Output

word_counts = build_word_frequency(messages)
print_top_words(word_counts, top_n=10)


  THIS GROUP'S FAVOURITE WORDS
  how        ████████████████████ 321
  guys       ████████████████████ 318
  about      █████████████████    274
  today      ████████████████     257
  just       █████████████        208
  which      █████████████        202
  everyone   ████████████         187
  telling    ███████████          179
  up         ███████████          172
  bhai       ██████████           160


In [162]:
# Feature 6: Response speed and silent streaks

def compute_response_times(messages, participants):
    """
    For a person P, when someone else sends a message and P is the next different sender,
    measure that time gap.
    """
    gaps = {}
    for person in participants:
        gaps[person] = []

    for i in range(1, len(messages)):
        prev_msg = messages[i - 1]
        curr_msg = messages[i]

        if prev_msg['sender'] != curr_msg['sender']:
            gap_minutes = (curr_msg['datetime'] - prev_msg['datetime']).total_seconds() / 60
            gaps[curr_msg['sender']].append(gap_minutes)

    avg_response = {}
    for person in participants:
        if gaps[person]:
            avg_response[person] = sum(gaps[person]) / len(gaps[person])
        else:
            avg_response[person] = None

    return avg_response, gaps


def build_person_day_activity(messages, participants, start_date, total_days):
    activity = {}
    for person in participants:
        activity[person] = {}
        for offset in range(total_days):
            current_day = start_date + timedelta(days=offset)
            activity[person][current_day] = 0

    for msg in messages:
        activity[msg['sender']][msg['date']] += 1

    return activity


def longest_silent_streak_for_person(day_map):
    days = sorted(day_map.keys())
    longest = 0
    longest_start = None
    longest_end = None

    current = 0
    current_start = None

    for day in days:
        if day_map[day] == 0:
            if current == 0:
                current_start = day
            current += 1
        else:
            if current > longest:
                longest = current
                longest_start = current_start
                longest_end = day - timedelta(days=1)
            current = 0
            current_start = None

    if current > longest:
        longest = current
        longest_start = current_start
        longest_end = days[-1]

    return longest, longest_start, longest_end


def compute_silent_streaks(messages, participants):
    first_dt, last_dt, total_days = get_date_range(messages)
    activity = build_person_day_activity(messages, participants, first_dt.date(), total_days)

    streaks = {}
    for person in participants:
        streak_len, start_day, end_day = longest_silent_streak_for_person(activity[person])
        active_days = 0
        for day in activity[person]:
            if activity[person][day] > 0:
                active_days += 1

        streaks[person] = {
            'longest_streak': streak_len,
            'start_day': start_day,
            'end_day': end_day,
            'active_days': active_days,
            'silent_days': total_days - active_days,
            'total_days': total_days
        }

    return streaks


def print_response_and_streaks(messages, participants):
    avg_response, _ = compute_response_times(messages, participants)
    streaks = compute_silent_streaks(messages, participants)

    valid_response = {k: v for k, v in avg_response.items() if v is not None}
    fastest = min(valid_response, key=valid_response.get)
    slowest = max(valid_response, key=valid_response.get)

    print_divider()
    print("  RESPONSE PATTERNS")
    print_divider()
    print(f"  Fastest replier : {fastest:<8} ({minutes_to_readable(avg_response[fastest])})")
    print(f"  Slowest replier : {slowest:<8} ({minutes_to_readable(avg_response[slowest])})")
    print()
    print("  LONGEST SILENT STREAKS")

    ordered = sorted(streaks.items(), key=lambda x: x[1]['longest_streak'], reverse=True)

    for person, info in ordered:
        streak_len = info['longest_streak']
        if info['start_day'] and info['end_day']:
            if streak_len > 0:
                date_part = f" ({format_short_date(datetime.combine(info['start_day'], datetime.min.time()))} - {format_short_date(datetime.combine(info['end_day'], datetime.min.time()))})"
            else:
                date_part = ""
        else:
            date_part = ""

        print(f"    {person:<8}: {streak_len:>2} days{date_part}")

    return avg_response, streaks


In [163]:
# Feature 6.1: Output

avg_response, silent_streaks = print_response_and_streaks(messages, participants)


  RESPONSE PATTERNS
  Fastest replier : Rahul    (34.9 minutes)
  Slowest replier : Aman     (55.4 minutes)

  LONGEST SILENT STREAKS
    Vikas   : 11 days (23 Apr - 03 May)
    Aman    :  0 days
    Karan   :  0 days
    Neha    :  0 days
    Priya   :  0 days
    Rahul   :  0 days


In [164]:
# Feature 7: Message style analysis

def count_emojis_like(text):
    """
    Simple heuristic: count non-ascii characters as possible emojis/symbols.
    Dataset mostly won't contain emojis, but this supports bonus real chats.
    """
    count = 0
    for ch in text:
        if ord(ch) > 127:
            count += 1
    return count


def analyze_message_style(messages, participants):
    stats = {}
    for person in participants:
        stats[person] = {
            'message_count': 0,
            'text_message_count': 0,
            'total_chars': 0,
            'total_words': 0,
            'question_count': 0,
            'exclamation_messages': 0,
            'emoji_count': 0,
            'all_caps_messages': 0
        }

    for msg in messages:
        person = msg['sender']
        text = msg['text']
        stats[person]['message_count'] += 1

        if text.endswith('?'):
            stats[person]['question_count'] += 1

        if text.count('!') >= 2:
            stats[person]['exclamation_messages'] += 1

        if not msg['is_media'] and not msg['is_deleted']:
            stats[person]['text_message_count'] += 1
            stats[person]['total_chars'] += len(text)
            word_count = len(text.split())
            stats[person]['total_words'] += word_count
            stats[person]['emoji_count'] += count_emojis_like(text)

            alpha_chars = [ch for ch in text if ch.isalpha()]
            if len(alpha_chars) >= 3 and text.isupper():
                stats[person]['all_caps_messages'] += 1

    for person in participants:
        text_count = stats[person]['text_message_count']
        if text_count > 0:
            stats[person]['avg_msg_length_chars'] = stats[person]['total_chars'] / text_count
            stats[person]['avg_words_per_msg'] = stats[person]['total_words'] / text_count
        else:
            stats[person]['avg_msg_length_chars'] = 0
            stats[person]['avg_words_per_msg'] = 0

    return stats


def print_message_style_stats(style_stats, participants):
    print_divider()
    print("  MESSAGE STYLE ANALYSIS")
    print_divider()

    for person in participants:
        s = style_stats[person]
        print(f"  {person}")
        print(f"    Avg message length : {s['avg_msg_length_chars']:.1f} chars")
        print(f"    Avg words/message  : {s['avg_words_per_msg']:.1f}")
        print(f"    Total characters   : {s['total_chars']}")
        print(f"    Questions          : {s['question_count']}")
        print(f"    Exclamation msgs   : {s['exclamation_messages']}")
        print(f"    Emoji-like count   : {s['emoji_count']}")
        print()


In [165]:
# Feature 7.1: Output

style_stats = analyze_message_style(messages, participants)
print_message_style_stats(style_stats, participants)


  MESSAGE STYLE ANALYSIS
  Aman
    Avg message length : 26.7 chars
    Avg words/message  : 5.0
    Total characters   : 12934
    Questions          : 32
    Exclamation msgs   : 0
    Emoji-like count   : 0

  Karan
    Avg message length : 311.4 chars
    Avg words/message  : 57.0
    Total characters   : 107437
    Questions          : 0
    Exclamation msgs   : 0
    Emoji-like count   : 0

  Neha
    Avg message length : 25.4 chars
    Avg words/message  : 5.3
    Total characters   : 15831
    Questions          : 39
    Exclamation msgs   : 32
    Emoji-like count   : 0

  Priya
    Avg message length : 28.2 chars
    Avg words/message  : 5.0
    Total characters   : 20082
    Questions          : 210
    Exclamation msgs   : 0
    Emoji-like count   : 0

  Rahul
    Avg message length : 10.7 chars
    Avg words/message  : 2.6
    Total characters   : 10048
    Questions          : 36
    Exclamation msgs   : 0
    Emoji-like count   : 0

  Vikas
    Avg message length : 7.2 c

In [166]:
# Feature 8: Personality archetype system

CARING_KEYWORDS = [
    'okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',
    'reminder', 'drink water', "don't forget", 'rest', 'careful'
]

COMEDY_WORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']

def get_messages_by_person(messages, participants):
    grouped = {}
    for person in participants:
        grouped[person] = []
    for msg in messages:
        grouped[msg['sender']].append(msg)
    return grouped


def average_consecutive_burst(messages, person):
    bursts = []
    current_burst = 0

    for msg in messages:
        if msg['sender'] == person:
            current_burst += 1
        else:
            if current_burst > 0:
                bursts.append(current_burst)
                current_burst = 0

    if current_burst > 0:
        bursts.append(current_burst)

    if not bursts:
        return 0
    return sum(bursts) / len(bursts)


def caring_keyword_score(person_messages):
    score = 0
    for msg in person_messages:
        text = msg['text'].lower()
        for keyword in CARING_KEYWORDS:
            score += text.count(keyword)
    return score


def night_owl_ratio(person_messages):
    if not person_messages:
        return 0
    night_count = 0
    for msg in person_messages:
        hour = msg['hour']
        if hour >= 23 or hour <= 4:
            night_count += 1
    return night_count / len(person_messages)


def drama_ratio(person_messages):
    if not person_messages:
        return 0

    drama_count = 0
    for msg in person_messages:
        text = msg['text']

        alpha_chars = [ch for ch in text if ch.isalpha()]
        is_caps = len(alpha_chars) >= 3 and text.isupper()
        has_exclaim = text.count('!') >= 2

        if is_caps or has_exclaim:
            drama_count += 1

    return drama_count / len(person_messages)


def ghost_ratio(streak_info):
    return streak_info['silent_days'] / streak_info['total_days']


def comedian_ratio(person_messages):
    if not person_messages:
        return 0

    comedy_hits = 0
    for msg in person_messages:
        text = msg['text'].lower()
        for word in COMEDY_WORDS:
            comedy_hits += text.count(word)

    return comedy_hits / len(person_messages)


def question_ratio(person_messages):
    if not person_messages:
        return 0

    q = 0
    for msg in person_messages:
        if msg['text'].strip().endswith('?'):
            q += 1
    return q / len(person_messages)


def build_archetype_scores(messages, participants, style_stats, silent_streaks):
    grouped = get_messages_by_person(messages, participants)
    scores = {}

    for person in participants:
        person_messages = grouped[person]

        scores[person] = {
            'THE SPAMMER': average_consecutive_burst(messages, person),
            'THE GROUP MOM': caring_keyword_score(person_messages),
            'THE NIGHT OWL': night_owl_ratio(person_messages),
            'THE STORYTELLER': style_stats[person]['avg_words_per_msg'],
            'THE DRAMA QUEEN': drama_ratio(person_messages),
            'THE GHOST': ghost_ratio(silent_streaks[person]),
            'THE COMEDIAN': comedian_ratio(person_messages),
            'THE QUESTION MASTER': question_ratio(person_messages)
        }

    return scores


def assign_unique_archetypes(archetype_scores):
    """
    Exclusive assignment:
    - For each archetype, rank people by score
    - Greedy assignment from strongest score overall
    """
    assignments = {}
    used_people = set()

    all_candidates = []
    for person in archetype_scores:
        for archetype, score in archetype_scores[person].items():
            all_candidates.append((score, person, archetype))

    all_candidates.sort(reverse=True, key=lambda x: x[0])

    used_archetypes = set()

    for score, person, archetype in all_candidates:
        if person not in used_people and archetype not in used_archetypes:
            assignments[person] = (archetype, score)
            used_people.add(person)
            used_archetypes.add(archetype)

    # Fallback in case a person remains unassigned
    for person in archetype_scores:
        if person not in assignments:
            best_archetype = max(archetype_scores[person], key=archetype_scores[person].get)
            assignments[person] = (best_archetype, archetype_scores[person][best_archetype])

    return assignments


def describe_archetype(person, archetype, score, archetype_scores):
    if archetype == 'THE SPAMMER':
        return f"avg burst {score:.1f} msgs"
    elif archetype == 'THE GROUP MOM':
        return f"caring score {int(score)}"
    elif archetype == 'THE NIGHT OWL':
        return f"{score * 100:.1f}% msgs at night"
    elif archetype == 'THE STORYTELLER':
        return f"avg {score:.1f} words/msg"
    elif archetype == 'THE DRAMA QUEEN':
        return f"{score * 100:.1f}% dramatic msgs"
    elif archetype == 'THE GHOST':
        return f"silent {score * 100:.1f}% of days"
    elif archetype == 'THE COMEDIAN':
        return f"humor ratio {score:.3f}"
    elif archetype == 'THE QUESTION MASTER':
        return f"{score * 100:.1f}% questions"
    return f"score {score:.2f}"


def print_archetypes(assignments, archetype_scores):
    print_divider()
    print("  PERSONALITY ARCHETYPES")
    print_divider()

    for person in sorted(assignments.keys()):
        archetype, score = assignments[person]
        desc = describe_archetype(person, archetype, score, archetype_scores)
        print(f"  {person:<8} -> {archetype:<20} ({desc})")


In [167]:
# Feature 8.1: Output

archetype_scores = build_archetype_scores(messages, participants, style_stats, silent_streaks)
assignments = assign_unique_archetypes(archetype_scores)
print_archetypes(assignments, archetype_scores)


  PERSONALITY ARCHETYPES
  Aman     -> THE NIGHT OWL        (79.8% msgs at night)
  Karan    -> THE STORYTELLER      (avg 57.0 words/msg)
  Neha     -> THE DRAMA QUEEN      (62.2% dramatic msgs)
  Priya    -> THE GROUP MOM        (caring score 679)
  Rahul    -> THE SPAMMER          (avg burst 4.5 msgs)
  Vikas    -> THE GHOST            (silent 73.3% of days)


In [168]:
# Feature 9 helper: fun insights

def find_most_active_member(message_counts):
    return max(message_counts, key=message_counts.get)


def find_night_owl(participants, messages):
    grouped = get_messages_by_person(messages, participants)
    best_person = None
    best_ratio = -1

    for person in participants:
        ratio = night_owl_ratio(grouped[person])
        if ratio > best_ratio:
            best_ratio = ratio
            best_person = person

    return best_person, best_ratio


def find_ghost_member(silent_streaks):
    best_person = None
    best_ratio = -1

    for person, info in silent_streaks.items():
        ratio = info['silent_days'] / info['total_days']
        if ratio > best_ratio:
            best_ratio = ratio
            best_person = person

    return best_person, best_ratio


In [169]:
# Feature 9: Final professional report

def print_final_report(group_name, messages, participants, heatmap, style_stats, archetype_scores, assignments):
    message_counts = count_messages_per_person(messages)
    sorted_counts = sort_counts_desc(message_counts)
    total_messages = len(messages)
    first_dt, last_dt, total_days = get_date_range(messages)
    busiest_day, busiest_day_count, _ = get_busiest_day(messages)
    busiest_hour, busiest_hour_count, _ = get_busiest_hour(messages)
    word_counts = build_word_frequency(messages)
    top_words = get_top_n_items(word_counts, 5)
    avg_response, silent_streaks = compute_response_times(messages, participants)[0], compute_silent_streaks(messages, participants)

    most_active_member = find_most_active_member(message_counts)
    night_owl_name, night_owl_pct = find_night_owl(participants, messages)
    ghost_name, ghost_pct = find_ghost_member(silent_streaks)

    valid_response = {k: v for k, v in avg_response.items() if v is not None}
    fastest = min(valid_response, key=valid_response.get)
    slowest = max(valid_response, key=valid_response.get)

    print_divider(60, '=')
    print(f'  GROUPDNA REPORT  -  "{group_name}"')
    print(f'  {total_days} days  -  {total_messages:,} messages  -  {len(participants)} members')
    print_divider(60, '=')
    print()

    print(f"  Period          : {format_date(first_dt)} to {format_date(last_dt)}")
    print(f"  Busiest day     : {format_date(datetime.combine(busiest_day, datetime.min.time()))}  ({busiest_day_count} messages)")
    print(f"  Busiest hour    : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00  ({busiest_hour_count} messages)")
    print()

    print("  MESSAGES PER PERSON")
    max_count = sorted_counts[0][1]
    for person, count in sorted_counts:
        pct = (count / total_messages) * 100
        bar = make_bar(count, max_count, width=20, char='█')
        print(f"    {person:<8} {bar:<20} {count:>4}  ({pct:>4.1f}%)")
    print()

    print("  ACTIVITY HEATMAP (hours 00 to 23)")
    print("         " + " ".join([f"{h:02d}" for h in range(24)]))
    for i, person in enumerate(participants):
        row = heatmap[i]
        row_max = int(np.max(row))
        symbols = [cell_to_symbol(v, row_max) for v in row]
        print(f"    {person:<8}" + " ".join(symbols))
    print()

    print("  THIS GROUP'S FAVOURITE WORDS")
    if top_words:
        top_max = top_words[0][1]
        for word, count in top_words:
            bar = make_bar(count, top_max, width=20, char='█')
            print(f"    {word:<8} {bar:<20} {count}")
    print()

    print("  RESPONSE PATTERNS")
    print(f"    Fastest replier : {fastest:<8} ({minutes_to_readable(avg_response[fastest])})")
    print(f"    Slowest replier : {slowest:<8} ({minutes_to_readable(avg_response[slowest])})")
    print()

    print("  LONGEST SILENT STREAKS")
    ordered_streaks = sorted(silent_streaks.items(), key=lambda x: x[1]['longest_streak'], reverse=True)
    for person, info in ordered_streaks:
        if info['longest_streak'] > 0 and info['start_day'] and info['end_day']:
            date_part = f" ({format_short_date(datetime.combine(info['start_day'], datetime.min.time()))} - {format_short_date(datetime.combine(info['end_day'], datetime.min.time()))})"
        else:
            date_part = ""
        print(f"    {person:<8}: {info['longest_streak']:>2} days{date_part}")
    print()

    print("  PERSONALITY ARCHETYPES")
    for person in sorted(assignments.keys()):
        archetype, score = assignments[person]
        desc = describe_archetype(person, archetype, score, archetype_scores)
        print(f"    {person:<8} -> {archetype:<20} ({desc})")
    print()

    print("  FUN INSIGHTS")
    print(f"    Most active member : {most_active_member}")
    print(f"    Night owl          : {night_owl_name} ({night_owl_pct * 100:.1f}% night messages)")
    print(f"    Ghost member       : {ghost_name} ({ghost_pct * 100:.1f}% silent days)")
    print(f"    System messages    : {system_count}")
    print(f"    Media shared       : {media_count}")
    print(f"    Deleted messages   : {deleted_count}")
    print()

    print_divider(60, '=')
    print("  Generated by GroupDNA  -  Built with Python + NumPy")
    print_divider(60, '=')


In [170]:
# Feature 9.1: Final output

print_final_report(
    group_name=group_name,
    messages=messages,
    participants=participants,
    heatmap=heatmap,
    style_stats=style_stats,
    archetype_scores=archetype_scores,
    assignments=assignments
)


  GROUPDNA REPORT  -  "Hostel Bois 4ever"
  60 days  -  3,174 messages  -  6 members

  Period          : 01 April 2024 to 30 May 2024
  Busiest day     : 04 May 2024  (76 messages)
  Busiest hour    : 18:00 - 19:00  (248 messages)

  MESSAGES PER PERSON
    Rahul    ████████████████████  953  (30.0%)
    Priya    ███████████████       718  (22.6%)
    Neha     █████████████         635  (20.0%)
    Aman     ██████████            490  (15.4%)
    Karan    ███████               354  (11.2%)
    Vikas    █                      24  ( 0.8%)

  ACTIVITY HEATMAP (hours 00 to 23)
         00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
    Aman    █ █ █ █ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . █
    Karan   . . . . . . . ░ ▒ ▒ █ ▒ █ █ █ █ █ █ █ █ █ ▒ ░ ░
    Neha    . . . . . ▒ ░ ░ █ █ █ ▒ █ █ ▒ ░ █ █ █ █ █ ▒ ▒ ▒
    Priya   . . . . . . ░ ▒ █ █ █ █ █ █ █ ▒ ▒ █ █ █ █ ▒ ▒ ░
    Rahul   ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █
    Vikas   . . . . . . . ▒ █ ▒ ▒ . █ █

## Analysis Summary

This report was generated by parsing the raw WhatsApp export and computing:

- participant activity
- message volume
- time-based behavior
- language frequency
- response behavior
- message style
- rule-based archetype classification

The project demonstrates how much can be built using only:
- strings
- lists
- dictionaries
- loops
- functions
- NumPy
- file I/O
- datetime

without using pandas or plotting libraries.
